# Lab 7: Markov decision processes and Q-learning

The first part plans in grid worlds whose transitions are known: policy evaluation by
solving the Bellman equations, value iteration, and the effect of the discount factor. The
second part learns without the model, with Q-learning, on the cliff walking problem.

Replace the lines `...  # TODO` with your code. Each implementation is followed by a cell of
automatic tests, which must print "all tests passed".

This lab needs the version of `seqlearn` that contains `mdp.py`: if the imports below fail,
download the setup archive again from the course page and replace the folder `seqlearn/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from seqlearn import checks
from seqlearn.mdp import ACTIONS, make_choice, make_cliff, make_maze
from seqlearn.plotting import COLORS, MUTED, plot_mean, set_style

set_style()

# Part I: Planning with a known model

## 1. The framework

A finite MDP is given by two arrays: `P[s, a, s2]`, the probability of moving from state
`s` to state `s2` with action `a`, and `R[s, a]`, the expected reward of this step. A
deterministic policy is an array `policy` with one action per state.

The maze below has a goal G (reward $+1$) and a trap T (reward $-1$), both terminal. The
actions are up, right, down and left; with probability $0.1$, the move goes in a random
direction instead (the floor is slippery). `maze.plot(ax, values, policy)` draws a value
function and a policy.

In [ ]:
maze = make_maze()
print(f"{maze.n_states} states, {maze.n_actions} actions: {ACTIONS}")
print("P:", maze.P.shape, " R:", maze.R.shape)
fig, ax = plt.subplots(figsize=(5, 3))
maze.plot(ax, title="Maze")
plt.show()

## 2. Policy evaluation

The value of a policy $\pi$ satisfies the Bellman equations
$V^\pi(s) = r(s, \pi(s)) + \gamma \sum_{s'} P(s' \mid s, \pi(s))\, V^\pi(s')$, that is,
$V^\pi = r_\pi + \gamma P_\pi V^\pi$ in matrix form, where $P_\pi$ is the $n \times n$ matrix
of the transitions under $\pi$ and $r_\pi$ the vector of the rewards. Hence
$V^\pi = (I - \gamma P_\pi)^{-1} r_\pi$ (Tutorial 7, exercise 1).

**Question 1.** Complete `policy_evaluation(P, R, policy, gamma)`.

*Hint:* `P[np.arange(n), policy]` is the matrix $P_\pi$, and `np.linalg.solve` solves a
linear system.

In [ ]:
def policy_evaluation(P, R, policy, gamma):
    ...  # TODO


checks.check_policy_evaluation(policy_evaluation)

**Question 2.** Evaluate, with $\gamma = 0.95$, the policy that always goes right and the
policy that always goes down, and draw their value functions. Print the value of the start
state for both.

In [ ]:
GAMMA = 0.95

In [ ]:
...  # TODO

## 3. Value iteration

Value iteration starts from $V_0 = 0$ and applies the Bellman optimality operator:
$$Q_{k+1}(s, a) = r(s, a) + \gamma \sum_{s'} P(s' \mid s, a)\, V_k(s') \enspace, \qquad
V_{k+1}(s) = \max_a Q_{k+1}(s, a) \enspace.$$
It converges to the optimal value $V^\star$, and the greedy policy
$s \mapsto \arg\max_a Q(s, a)$ is optimal.

**Question 3.** Complete `value_iteration(P, R, gamma, n_iter)`, which runs `n_iter`
iterations and returns the pair `(V, Q)` of the last iteration.

*Hint:* `P @ V` has shape `(n_states, n_actions)`: entry $(s, a)$ is
$\sum_{s'} P(s' \mid s, a) V(s')$.

In [ ]:
def value_iteration(P, R, gamma, n_iter):
    ...  # TODO


checks.check_value_iteration(value_iteration)

**Question 4.** On the maze, with $\gamma = 0.95$, draw the value function and the greedy
policy after $5$, $15$ and $200$ iterations. Then plot the error
$\|V_k - V^\star\|_\infty$ as a function of $k$, with a logarithmic y-axis, using
$V_{1000}$ as $V^\star$, and add the line $k \mapsto \gamma^k \|V^\star\|_\infty$.

In [ ]:
...  # TODO

## 4. The discount factor

**Question 5.** On the grid `make_choice()`, a small reward g ($0.1$) is $3$ moves from the
start S and a large one G ($1$) is $11$ moves away; the moves are deterministic. Draw the
optimal policy for $\gamma \in \{0.5, 0.7, 0.8, 0.95\}$, and find the threshold on $\gamma$
above which the agent goes to G.

In [ ]:
...  # TODO

# Part II: Learning without the model

## 5. Q-learning

In practice $P$ and $R$ are unknown: the agent only observes transitions
$(s, a, r, s')$ by playing. Q-learning updates, after each transition,
$$Q(s, a) \leftarrow Q(s, a) + \alpha \big(r + \gamma \max_{a'} Q(s', a') - Q(s, a)\big)
\enspace,$$
where the target is $r$ alone if $s'$ is terminal. It plays $\varepsilon$-greedy with
respect to $Q$, as the agents of Lab 1.

**Question 6.** Complete `QLearningAgent`: `act(s)` plays $\varepsilon$-greedy (the greedy
action with the smallest index in case of a tie), and `update(s, a, r, s_next, done)`
applies the update above.

In [ ]:
class QLearningAgent:
    def __init__(self, n_states, n_actions, alpha=0.5, gamma=1.0, epsilon=0.1, rng=None):
        self.alpha, self.gamma, self.epsilon = alpha, gamma, epsilon
        self.n_actions = n_actions
        self.rng = np.random.default_rng(rng)
        self.Q = np.zeros((n_states, n_actions))

    def act(self, s):
        ...  # TODO

    def update(self, s, a, r, s_next, done):
        ...  # TODO


checks.check_qlearning(QLearningAgent)

## 6. Cliff walking

In cliff walking (Sutton and Barto, example 6.6), each move costs $-1$, and stepping on the
cliff C costs $-100$ and sends the agent back to the start S. The episode ends at G. The
environment is simulated by `cliff.reset()`, which returns the start state, and
`cliff.step(a)`, which returns `(s_next, reward, done)`. There is no discount: $\gamma = 1$.

The function below runs an agent for a number of episodes and returns the sum of the
rewards of each episode.

In [ ]:
cliff = make_cliff(rng=0)
fig, ax = plt.subplots(figsize=(7, 2.5))
cliff.plot(ax, title="Cliff walking")
plt.show()


def train(env, agent, n_episodes, max_steps=1000):
    returns = np.empty(n_episodes)
    for episode in range(n_episodes):
        s, total = env.reset(), 0.0
        for _ in range(max_steps):
            a = agent.act(s)
            s_next, r, done = env.step(a)
            agent.update(s, a, r, s_next, done)
            s, total = s_next, total + r
            if done:
                break
        returns[episode] = total
    return returns

**Question 7.** Train $10$ agents (seeds $0$ to $9$) with $\alpha = 0.5$ and
$\varepsilon = 0.1$ for $500$ episodes each, and plot the mean return per episode, with
`plot_mean`. Draw the greedy policy of one agent at the end, and compare its return with the
optimal value $V^\star(\text{start})$ computed by value iteration.

In [ ]:
...  # TODO

**Question 8.** Plot the mean return over the last $100$ episodes, and the return of the
greedy policy at the end (one episode with $\varepsilon = 0$), as functions of
$\varepsilon \in \{0.01, 0.05, 0.1, 0.2, 0.4\}$, for $\alpha = 0.5$ and $10$ agents.

In [ ]:
...  # TODO

## Going further

**Question 9.** On the maze, which is stochastic, run Q-learning with $\gamma = 0.95$,
$\varepsilon = 0.1$ and $\alpha = 0.05$ for $3000$ episodes (at most $200$ moves each),
with seeds $0$ to $4$. Evaluate the greedy policy at the end with `policy_evaluation`, and
compare $V^\pi(\text{start})$ with the optimal value of Question 4. Then start instead
from $Q = 1$ on the nonterminal states (optimistic initialization), and compare.

In [ ]:
...  # TODO